In [1]:
import numpy as np
import tensorflow as tf

BETA = 0.25
EPS = 1e-8

In [2]:
class VectorQuantizer(tf.keras.layers.Layer):
    def __init__(self, num_codes, code_dim, beta=BETA):
        super().__init__()
        self.num_codes = num_codes
        self.code_dim = code_dim
        self.beta = beta
        init = tf.random_uniform_initializer(-1.0 / num_codes, 1.0 / num_codes)
        self.codebook = tf.Variable(init(shape=(num_codes, code_dim)), trainable=True, name="codebook")

    def call(self, z_e):
        b, h, w, c = tf.shape(z_e)[0], tf.shape(z_e)[1], tf.shape(z_e)[2], tf.shape(z_e)[3]
        flat = tf.reshape(z_e, (-1, self.code_dim))

        dist = (tf.reduce_sum(flat ** 2, axis=1, keepdims=True)
                - 2 * tf.matmul(flat, self.codebook, transpose_b=True)
                + tf.reduce_sum(self.codebook ** 2, axis=1)[None, :])
        indices = tf.argmin(dist, axis=1)
        z_q_flat = tf.gather(self.codebook, indices)
        z_q = tf.reshape(z_q_flat, tf.shape(z_e))

        codebook_loss = tf.reduce_mean((tf.stop_gradient(z_e) - z_q) ** 2)
        commitment_loss = tf.reduce_mean((z_e - tf.stop_gradient(z_q)) ** 2)
        vq_loss = codebook_loss + self.beta * commitment_loss

        z_q_st = z_e + tf.stop_gradient(z_q - z_e)

        indices_2d = tf.reshape(indices, (b, h, w))
        one_hot = tf.one_hot(indices, self.num_codes)
        avg_probs = tf.reduce_mean(one_hot, axis=0)
        perplexity = tf.exp(-tf.reduce_sum(avg_probs * tf.math.log(avg_probs + EPS)))
        utilization = tf.reduce_sum(tf.cast(avg_probs > 0, tf.float32)) / self.num_codes

        return z_q_st, vq_loss, indices_2d, perplexity, utilization


In [3]:
class SelfAttention2d(tf.keras.layers.Layer):
    def __init__(self, channels):
        super().__init__()
        self.q_conv = tf.keras.layers.Conv2D(channels, 1)
        self.k_conv = tf.keras.layers.Conv2D(channels, 1)
        self.v_conv = tf.keras.layers.Conv2D(channels, 1)
        self.out_conv = tf.keras.layers.Conv2D(channels, 1)
        self.gamma = tf.Variable(0.0, trainable=True, name="attn_gamma")

    def call(self, x):
        b, h, w, c = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2], tf.shape(x)[3]
        q = tf.reshape(self.q_conv(x), (b, h * w, c))
        k = tf.reshape(self.k_conv(x), (b, h * w, c))
        v = tf.reshape(self.v_conv(x), (b, h * w, c))

        scores = tf.matmul(q, k, transpose_b=True) / tf.sqrt(tf.cast(c, tf.float32))
        weights = tf.nn.softmax(scores, axis=-1)
        out = tf.matmul(weights, v)
        out = tf.reshape(out, (b, h, w, c))
        out = self.out_conv(out)
        return x + self.gamma * out

In [4]:
class Encoder(tf.keras.Model):
    def __init__(self, latent_ch=64, in_ch=3):
        super().__init__()
        self.conv1 = tf.keras.layers.Conv2D(64, 4, strides=2, padding="same", activation="relu")
        self.conv2 = tf.keras.layers.Conv2D(128, 4, strides=2, padding="same", activation="relu")
        self.attn = SelfAttention2d(128)
        self.conv3 = tf.keras.layers.Conv2D(latent_ch, 3, padding="same")

    def call(self, x):
        x = self.conv1(x)
        x = self.conv2(x)  # spatial resolution -> bottleneck (16x16 given a 64x64 input)
        x = self.attn(x)
        return self.conv3(x)

In [5]:
class Decoder(tf.keras.Model):
    def __init__(self, latent_ch=64, out_ch=3):
        super().__init__()
        self.conv0 = tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu")
        self.attn = SelfAttention2d(128)
        self.deconv1 = tf.keras.layers.Conv2DTranspose(64, 4, strides=2, padding="same", activation="relu")
        self.deconv2 = tf.keras.layers.Conv2DTranspose(out_ch, 4, strides=2, padding="same", activation="tanh")

    def call(self, z_q):
        x = self.conv0(z_q)
        x = self.attn(x)
        x = self.deconv1(x)
        return self.deconv2(x)

In [6]:
class PatchDiscriminator(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.conv1 = tf.keras.layers.Conv2D(64, 4, strides=2, padding="same")
        self.act1 = tf.keras.layers.LeakyReLU(0.2)
        self.conv2 = tf.keras.layers.Conv2D(128, 4, strides=2, padding="same")
        self.norm2 = tf.keras.layers.LayerNormalization()
        self.act2 = tf.keras.layers.LeakyReLU(0.2)
        self.conv3 = tf.keras.layers.Conv2D(1, 4, strides=1, padding="same")

    def call(self, x):
        x = self.act1(self.conv1(x))
        x = self.act2(self.norm2(self.conv2(x)))
        return self.conv3(x)

In [7]:
class VQGAN(tf.keras.Model):
    def __init__(self, latent_ch=64, num_codes=512, in_ch=3):
        super().__init__()
        self.encoder = Encoder(latent_ch, in_ch)
        self.vq = VectorQuantizer(num_codes, latent_ch)
        self.decoder = Decoder(latent_ch, in_ch)

    def call(self, x):
        z_e = self.encoder(x)
        z_q, vq_loss, indices, perplexity, utilization = self.vq(z_e)
        x_hat = self.decoder(z_q)
        return x_hat, vq_loss, perplexity, utilization

In [8]:
def train_vqgan(model, discriminator, dataset, epochs=5, lr=2e-4):
    g_opt = tf.keras.optimizers.Adam(lr, beta_1=0.5)
    d_opt = tf.keras.optimizers.Adam(lr, beta_1=0.5)
    bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

    for epoch in range(epochs):
        perp_hist, util_hist = [], []
        for real in dataset:
            with tf.GradientTape() as g_tape:
                x_hat, vq_loss, perplexity, utilization = model(real)
                rec_loss = tf.reduce_mean(tf.abs(real - x_hat))
                fake_logits = discriminator(x_hat)
                g_adv_loss = bce(tf.ones_like(fake_logits), fake_logits)
                g_loss = rec_loss + vq_loss + 0.1 * g_adv_loss
            g_grads = g_tape.gradient(g_loss, model.trainable_variables)
            g_opt.apply_gradients(zip(g_grads, model.trainable_variables))

            with tf.GradientTape() as d_tape:
                real_logits = discriminator(real)
                fake_logits = discriminator(tf.stop_gradient(x_hat))
                d_loss = (bce(tf.ones_like(real_logits), real_logits)
                          + bce(tf.zeros_like(fake_logits), fake_logits))
            d_grads = d_tape.gradient(d_loss, discriminator.trainable_variables)
            d_opt.apply_gradients(zip(d_grads, discriminator.trainable_variables))

            perp_hist.append(float(perplexity))
            util_hist.append(float(utilization))

        print(f"epoch {epoch + 1}/{epochs}  "
              f"perplexity={np.mean(perp_hist):.2f}  codebook_utilization={np.mean(util_hist) * 100:.1f}%")


In [9]:
model = VQGAN()
disc = PatchDiscriminator()
x = tf.random.normal((2, 64, 64, 3))
x_hat, vq_loss, perplexity, utilization = model(x)
print("recon:", x_hat.shape, "vq_loss:", float(vq_loss),
        "perplexity:", float(perplexity), "utilization:", float(utilization))
print("discriminator patch output:", disc(x_hat).shape)

recon: (2, 64, 64, 3) vq_loss: 0.018368123099207878 perplexity: 49.92236328125 utilization: 0.21484375
discriminator patch output: (2, 16, 16, 1)
